# Ensemble: Logistic Regression + XGBoost — CV ~0.917

Pipeline de dos modelos complementarios:
- **Logistic Regression** con pairwise target encoding + transformación logit3 (sklearn)
- **XGBoost** con soporte nativo de categorías + GPU

El ensemble por promedio simple supera a cada modelo individual.

| Modelo | CV AUC esperado |
|--------|----------------|
| Logistic Regression (pairwise TE + logit3) | ~0.916 |
| XGBoost (native cat, GPU) | ~0.9166 |
| **Ensemble** | **~0.917+** |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from itertools import combinations

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

print('Librerias cargadas')

## 1. Carga de Datos

In [ ]:
train_raw = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test_raw  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

y        = (train_raw['Churn'] == 'Yes').astype(int).values
ids_test = test_raw['id'].values

print(f'Train: {train_raw.shape[0]:,} filas x {train_raw.shape[1]} columnas')
print(f'Test:  {test_raw.shape[0]:,} filas x {test_raw.shape[1]} columnas')
print(f'Churn rate: {y.mean():.2%}')

## 2. Preprocessing Base (Label Encoding compartido train+test)

In [ ]:
DROP_COLS = ['id', 'customerID', 'Churn']

def label_encode_together(train_df, test_df):
    """Label encode en base al vocabulario combinado train+test."""
    train_df = train_df.copy()
    test_df  = test_df.copy()

    # Arreglar TotalCharges (puede contener strings vacios)
    for df in [train_df, test_df]:
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0.0)

    # Columnas a usar
    drop_train = [c for c in DROP_COLS if c in train_df.columns]
    drop_test  = [c for c in DROP_COLS if c in test_df.columns]
    X_tr = train_df.drop(columns=drop_train)
    X_te = test_df.drop(columns=drop_test)

    # Label encoding consistente: mapeo por vocabulario combinado
    cat_cols = X_tr.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        vocab = pd.concat([X_tr[col], X_te[col]], ignore_index=True).unique()
        mapping = {v: i for i, v in enumerate(sorted(str(v) for v in vocab))}
        X_tr[col] = X_tr[col].astype(str).map(mapping).astype('int32')
        X_te[col] = X_te[col].astype(str).map(mapping).astype('int32')

    return X_tr, X_te


X_base_train, X_base_test = label_encode_together(train_raw, test_raw)
FEATURES = X_base_train.columns.tolist()

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'X_base_train: {X_base_train.shape}')
print(f'X_base_test:  {X_base_test.shape}')

## 3. Modelo 1: Logistic Regression — Pairwise Target Encoding + Logit3

Para cada par de features (C(19,2) = 171 pares):
1. Concatenar valores como string → columna combinada
2. `TargetEncoder` (sklearn) → probabilidad media de churn por categoría combinada
3. Transformación logit3: `z = logit(x)`, `[z, z², z³]` → 513 features
4. `LogisticRegression(C=0.5)` con regularización L2

In [ ]:
VER_LR = 1
N_FOLDS = 5
EPS = 1e-6

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

PAIRS = list(combinations(FEATURES, 2))
print(f'Pares de features: {len(PAIRS)}  →  {len(PAIRS) * 3} features tras logit3')

X_tr_np = X_base_train.values
X_te_np = X_base_test.values


def build_pair_features(X_tr, X_te, feat_names, y_tr):
    """Construye 513 features: pairwise TargetEncoding + logit3."""
    pairs = list(combinations(range(len(feat_names)), 2))
    tr_cols, te_cols = [], []

    for i, j in pairs:
        # Crear columna combinada como string
        col_tr = X_tr[:, i].astype(str) + '__x__' + X_tr[:, j].astype(str)
        col_te = X_te[:, i].astype(str) + '__x__' + X_te[:, j].astype(str)

        # TargetEncoder: fit en train, transform train+test
        te = TargetEncoder(target_type='binary', smooth='auto', cv=5)
        enc_tr = te.fit_transform(col_tr.reshape(-1, 1), y_tr).ravel()
        enc_te = te.transform(col_te.reshape(-1, 1)).ravel()

        tr_cols.append(enc_tr)
        te_cols.append(enc_te)

    X_tr_pairs = np.column_stack(tr_cols)  # (n_train_fold, 171)
    X_te_pairs = np.column_stack(te_cols)  # (n_test, 171)

    # Logit3: z, z^2, z^3
    def logit3(arr):
        arr = np.clip(arr, EPS, 1 - EPS)
        z = np.log(arr / (1 - arr))
        return np.hstack([z, z**2, z**3])

    return logit3(X_tr_pairs), logit3(X_te_pairs)


oof_lr       = np.zeros(len(X_tr_np), dtype=np.float32)
test_sum_lr  = np.zeros(len(X_te_np), dtype=np.float32)
fold_aucs_lr = []

print(f'\n[logit_xgb_v{VER_LR}] Entrenando Logistic Regression...')

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr_np, y), 1):
    X_trn, X_val = X_tr_np[trn_idx], X_tr_np[val_idx]
    y_trn, y_val = y[trn_idx], y[val_idx]

    # Construir features de pares (target encoding fit solo en train fold)
    # Para el test: pasamos X_te_np completo, se transforma igual
    X_trn_feat, X_val_feat = build_pair_features(
        X_trn, X_val, FEATURES, y_trn
    )
    _, X_te_feat = build_pair_features(
        X_trn, X_te_np, FEATURES, y_trn
    )

    # Escalar
    scaler = StandardScaler()
    X_trn_feat = scaler.fit_transform(X_trn_feat)
    X_val_feat = scaler.transform(X_val_feat)
    X_te_feat  = scaler.transform(X_te_feat)

    # Fit
    model_lr = LogisticRegression(C=0.5, max_iter=4000, random_state=42, n_jobs=-1)
    model_lr.fit(X_trn_feat, y_trn)

    # OOF
    p_val = model_lr.predict_proba(X_val_feat)[:, 1]
    oof_lr[val_idx] = p_val

    # Test
    test_sum_lr += model_lr.predict_proba(X_te_feat)[:, 1]

    auc = roc_auc_score(y_val, p_val)
    fold_aucs_lr.append(auc)
    print(f'  Fold {fold}/{N_FOLDS}  AUC = {auc:.6f}')

test_preds_lr = (test_sum_lr / N_FOLDS).astype(np.float32)
oof_auc_lr    = roc_auc_score(y, oof_lr)

print(f'\n[logit_xgb_v{VER_LR}] OOF AUC: {oof_auc_lr:.6f}')
print(f'[logit_xgb_v{VER_LR}] Fold AUCs: {[round(a, 6) for a in fold_aucs_lr]}')

## 4. Modelo 2: XGBoost — Native Categorical + GPU

- `enable_categorical=True`: XGBoost maneja las categorías nativamente sin encoding manual
- `device='cuda'`: entrenamiento en GPU
- `early_stopping_rounds=200`: para automáticamente cuando el AUC de validación no mejora

In [ ]:
VER_XGB = 1


def preprocess_xgb(train_df, test_df):
    """Dtype categórico compartido para XGBoost nativo."""
    train_df = train_df.copy()
    test_df  = test_df.copy()

    drop_train = [c for c in DROP_COLS if c in train_df.columns]
    drop_test  = [c for c in DROP_COLS if c in test_df.columns]
    X_tr = train_df.drop(columns=drop_train)
    X_te = test_df.drop(columns=drop_test)

    # TotalCharges → numérico
    for df in [X_tr, X_te]:
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0.0)

    # Dtype categórico compartido (vocabulario de ambos splits)
    all_combined = pd.concat([X_tr, X_te], ignore_index=True)
    cat_cols = all_combined.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        cats = pd.CategoricalDtype(categories=sorted(all_combined[col].dropna().unique()), ordered=False)
        X_tr[col] = X_tr[col].astype(cats)
        X_te[col] = X_te[col].astype(cats)

    return X_tr, X_te


X_xgb_train, X_xgb_test = preprocess_xgb(train_raw, test_raw)

xgb_params = dict(
    n_estimators       = 100_000,
    learning_rate      = 0.1,
    max_depth          = 3,
    min_child_weight   = 5,
    subsample          = 0.85,
    colsample_bytree   = 0.85,
    objective          = 'binary:logistic',
    eval_metric        = 'auc',
    tree_method        = 'hist',
    device             = 'cuda',
    enable_categorical = True,
    early_stopping_rounds = 200,
    random_state       = 42,
    n_jobs             = -1,
)

oof_xgb       = np.zeros(len(X_xgb_train), dtype=np.float32)
test_sum_xgb  = np.zeros(len(X_xgb_test),  dtype=np.float32)
fold_aucs_xgb = []

print(f'[logit_xgb_v{VER_XGB}] Entrenando XGBoost...')

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_xgb_train, y), 1):
    X_trn = X_xgb_train.iloc[trn_idx]
    X_val = X_xgb_train.iloc[val_idx]
    y_trn, y_val = y[trn_idx], y[val_idx]

    model_xgb = XGBClassifier(**xgb_params)
    model_xgb.fit(
        X_trn, y_trn,
        eval_set=[(X_val, y_val)],
        verbose=500,
    )

    p_val = model_xgb.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx] = p_val
    test_sum_xgb += model_xgb.predict_proba(X_xgb_test)[:, 1]

    auc = roc_auc_score(y_val, p_val)
    fold_aucs_xgb.append(auc)
    print(f'  Fold {fold}/{N_FOLDS}  AUC = {auc:.6f}  | best iteration: {model_xgb.best_iteration}')

test_preds_xgb = (test_sum_xgb / N_FOLDS).astype(np.float32)
oof_auc_xgb    = roc_auc_score(y, oof_xgb)

print(f'\n[logit_xgb_v{VER_XGB}] OOF AUC: {oof_auc_xgb:.6f}')
print(f'[logit_xgb_v{VER_XGB}] Fold AUCs: {[round(a, 6) for a in fold_aucs_xgb]}')

## 5. Ensemble — Promedio Simple

In [ ]:
oof_ensemble  = (oof_lr + oof_xgb) / 2
test_ensemble = (test_preds_lr + test_preds_xgb) / 2

oof_auc_ensemble = roc_auc_score(y, oof_ensemble)

print('=' * 55)
print('RESUMEN DE SCORES (OOF AUC)')
print('=' * 55)
print(f'  Logistic Regression:  {oof_auc_lr:.6f}')
print(f'  XGBoost:              {oof_auc_xgb:.6f}')
print(f'  Ensemble:             {oof_auc_ensemble:.6f}  ← submission')
print('=' * 55)

## 6. Submission

In [ ]:
submission = pd.DataFrame({
    'id':    ids_test,
    'Churn': test_ensemble.round(4),
})

submission_path =  '/kaggle/working/3_LR_XGB_ensemble_submission.csv'
submission.to_csv(submission_path, index=False)

print(f'Submission guardada: {submission_path}')
print(f'Filas:    {len(submission):,}')
print(f'Columnas: {list(submission.columns)}')
print(submission.head(5).to_string(index=False))

In [ ]:
!kaggle competitions submit -c playground-series-s6e3 \
  -f  /kaggle/working/3_LR_XGB_ensemble_submission.csv \
  -m "Ensemble LR (pairwise TE + logit3) + XGBoost (native cat, GPU) | CV {oof_auc_ensemble:.4f}"